# HW4 B: Automatic speech recognition
Build a **HuBERT → downsampling convolution → bidirectional LSTM → CTC**
recognizer on the supplied LibriSpeech subset. Speakers are disjoint across
train, validation, and test sets; test transcripts are withheld.

**33 points:** 12 for code (lengths 2, loss 3, decoding 2, model 5), 18 for test
word error rate (WER), and 3 for report B1. Code credit is independent of training progress.
For performance, use the first matching row:

| Test corpus WER | Points |
|---|---:|
| ≤50% | 18 |
| ≤60% | 15 |
| ≤70% | 12 |
| ≤80% | 9 |
| ≤90% | 6 |
| ≤99% | 3 |
| Otherwise | 0 |

Missing predictions count as deletions. Select checkpoints using validation WER.

**Report:** use the [template](https://github.com/Louis0324/eecs183-hw4/blob/main/report_template.txt)
for one selectable-text `report.pdf` covering **A1, A2, A3, B1, C1**. Put conceptual
answers in the PDF: aim for **3–5 sentences total per section**, plus requested
figures/sequences. Retain completed answers; mark unfinished items `Not completed`.


## Setup
Save a copy in Drive, select a Colab GPU, and run cells in order. Setup and data
downloads are automatic; no Hugging Face login is needed. If pip requests a
restart, restart and rerun setup.

Implement the TODOs using their docstrings. Keep the specified top-level
function/class names and signatures. Put extra imports and constants inside
functions/classes: the grader does not execute other top-level code.

Set `SMOKE_TEST=True` in setup for a quick check. For full experiments, set it to
`False`, choose a fresh `RESULTS` folder, and rerun from setup. Save your work to
Drive or download it before the runtime ends; Colab local storage is temporary.

1. Implement `ctc_output_lengths`, `ctc_loss`, `greedy_decode`, and the missing
   `ASR` methods.
2. Run the supplied training loop and choose the best validation checkpoint.
3. Export all 95 test predictions, answer B1, and run the final submission cell.

Data loading, optimizer settings, training, checkpoint saving/resume, and export
are supplied. The HuBERT convolutional frontend is frozen; the remaining HuBERT
layers and new CTC layers are trainable by default.


In [ ]:
REQUIREMENTS = 'requirements.txt'
# Colab: select Runtime > Change runtime type > GPU before training.
import sys, subprocess
from pathlib import Path

# Course release settings are supplied by staff; no student login is needed.
COURSE_REPO_ID = 'Louis0324/eecs183-hw4'
COURSE_REVISION = 'hw4-2026-v4'
import os
os.environ.setdefault("HW4_HF_REPO", COURSE_REPO_ID)
os.environ.setdefault("HW4_HF_REVISION", COURSE_REVISION)

def ensure_hw4_support(repo_id=None, revision='hw4-2026-v4', packager_only=False):
    """Fetch the public support bundle, or use a local/uploaded offline copy."""
    import hashlib
    from io import BytesIO
    import json
    from pathlib import Path
    import re
    from urllib.parse import quote
    from urllib.request import urlopen
    import zipfile

    required = 'make_submission.py' if packager_only else 'course_download.py'
    if Path(required).is_file():
        return
    local_bundle = Path('hw4-student.zip')
    if local_bundle.is_file():
        content = local_bundle.read_bytes()
    elif repo_id:
        if not re.fullmatch(r'[\w.-]+/[\w.-]+', repo_id):
            raise ValueError('Expected a Hugging Face repository ID like COURSE_ORG/eecs183-hw4.')
        base = f'https://huggingface.co/datasets/{repo_id}/resolve/{quote(revision, safe="")}/'
        print('Downloading the small HW4 support bundle from Hugging Face; no login is required.')
        try:
            with urlopen(base + 'release_manifest.json', timeout=30) as response:
                manifest = json.loads(response.read(1024 * 1024))
            spec = manifest['files']['hw4-student.zip']
            if not 0 < spec['bytes'] < 30 * 1024 * 1024:
                raise ValueError('Invalid support bundle size in the release manifest.')
            with urlopen(base + 'hw4-student.zip', timeout=30) as response:
                content = response.read(spec['bytes'] + 1)
            if len(content) != spec['bytes'] or hashlib.sha256(content).hexdigest() != spec['sha256']:
                raise ValueError('Support bundle checksum mismatch; rerun to download a fresh copy.')
        except Exception as error:
            raise RuntimeError('Could not download the HW4 support release. Rerun this cell; '
                               'if needed, download hw4-student.zip from the course Hugging Face repo '
                               'and upload it through the Files sidebar before retrying.') from error
    else:
        # Allows local staff testing and offline/manual use before publication.
        try:
            from google.colab import files
        except ImportError as error:
            raise FileNotFoundError('Put hw4-student.zip in this folder or configure the course Hugging Face repository.') from error
        print('Upload hw4-student.zip to load the supplied helpers.')
        uploaded = files.upload()
        archives = [value for name, value in uploaded.items() if name.lower().endswith('.zip')]
        if len(archives) != 1:
            raise ValueError('Upload exactly one hw4-student.zip support bundle, then rerun.')
        content = archives[0]

    with zipfile.ZipFile(BytesIO(content)) as archive:
        if archive.testzip() is not None:
            raise ValueError('Corrupt support ZIP; download a fresh copy.')
        for item in archive.infolist():
            path = Path(item.filename)
            if (path.is_absolute() or '..' in path.parts or
                    ((item.external_attr >> 16) & 0o170000) == 0o120000 or
                    not path.resolve().is_relative_to(Path.cwd().resolve())):
                raise ValueError('Unsafe support archive path.')
            # Browser/local notebook edits must never be replaced by starter copies.
            if item.is_dir() or path.suffix == '.ipynb' or (packager_only and item.filename != required):
                continue
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(archive.read(item))
    if not Path(required).is_file():
        raise ValueError(f'This ZIP is missing {required}; use the current hw4-student.zip support bundle.')

ensure_hw4_support(COURSE_REPO_ID, COURSE_REVISION)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', REQUIREMENTS])
import torch
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SMOKE_TEST = False  # True: only a few steps, for debugging your implementation.
DATA_ROOT = Path(os.environ.get('HW4_DATA_ROOT', 'data'))
RESULTS = Path('results'); RESULTS.mkdir(exist_ok=True)
# Checkpoints/results disappear when Colab deletes a runtime. Copy them to Drive
# or download them before disconnecting. Submission ZIPs contain no checkpoints.

In [ ]:
# Download only Part B data; verified cached copies are reused.
from course_download import ensure_course_data
DATA_ARCHIVE = None  # Optional offline override: Path("hw4-data-b.zip")
ensure_course_data('b', data_root=DATA_ROOT, archive=DATA_ARCHIVE)


In [ ]:
import json, math, copy
import numpy as np
import torch
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence, unpad_sequence
from sklearn.metrics import accuracy_score, r2_score
import matplotlib.pyplot as plt
from hw4_runtime import load_phoneme_data, load_mfcc_data, load_audio
np.random.seed(183); torch.manual_seed(183)
device = DEVICE

from hw4_runtime import CharacterTokenizer, SpeechTextDataset, train_asr_epoch, evaluate_asr, save_checkpoint
DATA_DIR = DATA_ROOT / 'asr'
dataset = SpeechTextDataset(DATA_DIR, 'train')
print('Training utterances:', len(dataset)); print(dataset[0]['text'])

## CTC building blocks
CTC sums over alignments between frames and text. Compute valid encoded lengths
per utterance; exclude audio/target padding.

- HuBERT convolution kernels: `[10,3,3,3,3,2,2]`; strides: `[5,2,2,2,2,2,2]`; no padding.
- Added convolution: kernel 2, stride 2. Each stage gives `floor((L-kernel)/stride)+1`.
- Apply log-softmax and transpose `[batch,time,vocab]` to `[time,batch,vocab]` for CTC.
- Blank ID: 0; padding ID: 1. Loss must be finite and preserve gradients.

Decoder check: `[blank,B,B,blank,B,A,A]` gives `BBA`; frames beyond the valid
length must not change the result.


In [ ]:
def ctc_output_lengths(waveform_lengths):
    """Compute valid frame counts after all seven HuBERT convolutions and the kernel-2/stride-2 downsampler. Input: int tensor [B] of waveform lengths; output: int64 [B], clamped at zero."""
    raise NotImplementedError('TODO: ctc_output_lengths')

In [ ]:
def ctc_loss(logits, targets, input_lengths, target_lengths, blank_id=0):
    """Input unnormalized logits [B,T,V], padded target IDs [B,U], input_lengths [B], target_lengths [B]. Return scalar mean torch CTC loss with blank_id and zero_infinity=True. Apply log_softmax, transpose, preserve gradients. Padding targets are excluded by target_lengths."""
    raise NotImplementedError('TODO: ctc_loss')

In [ ]:
def greedy_decode(logits, lengths, tokenizer):
    """Argmax logits [B,T,V], truncate each row to lengths[b], collapse adjacent repeated IDs, THEN remove blank/pad through tokenizer.decode. Return a list of B strings. Blanks separate repeated letters: A blank A -> AA."""
    raise NotImplementedError('TODO: greedy_decode')

## Implement the model
Use the supplied HuBERT model with its convolutional frontend frozen. The default
LSTM has 128 hidden units and two bidirectional layers. `encode` returns raw
`[batch,time,vocab]` logits and valid lengths; use packed sequences to exclude
padding from backward LSTM states. If you choose another HuBERT layer, record it
in the report summary.


In [ ]:
from hw4_runtime import load_hubert
class ASR(nn.Module):
    def __init__(self, upstream=None, hidden_size=128, num_layers=2):
        super().__init__()
        self.token_processor = CharacterTokenizer()
        # Pass an upstream explicitly for small offline tests.
        self.upstream = upstream if upstream is not None else load_hubert()
        self.upstream.feature_extractor._freeze_parameters()
        # TODO: create downsample Conv1d(768, 128, kernel_size=2, stride=2),
        # a batch_first bidirectional LSTM, and a final Linear to vocab size.
        raise NotImplementedError('TODO: ASR layers')

    def encode(self, input_values, attention_mask):
        """Return (raw logits [B,T,V], valid lengths [B]).
        Run HuBERT, transpose for Conv1d and back, run the LSTM and output layer.
        Use packed sequences for a bidirectional LSTM so padding cannot affect
        backward states. Calculate lengths with ctc_output_lengths.
        """
        raise NotImplementedError('TODO: ASR.encode')

    def forward(self, input_values, text, attention_mask=None):
        if attention_mask is None:
            attention_mask = torch.ones_like(input_values, dtype=torch.long)
        logits, lengths = self.encode(input_values, attention_mask)
        targets, target_lengths = self.token_processor.encode_torch_batch(text)
        return ctc_loss(logits, targets.to(input_values.device), lengths, target_lengths)

    @torch.no_grad()
    def predict(self, input_values, attention_mask=None):
        if attention_mask is None:
            attention_mask = torch.ones_like(input_values, dtype=torch.long)
        logits, lengths = self.encode(input_values, attention_mask)
        return greedy_decode(logits, lengths, self.token_processor), logits

## Train and choose a checkpoint
Start with batch size 2 on a T4; reduce it or freeze more of HuBERT if needed.
The supplied optimizer uses a smaller learning rate for HuBERT than for new
layers. Early blank-only predictions and 100% WER are common; check valid lengths
and finite/improving loss before diagnosing a bug.

Checkpoints save at epoch end; a runtime reset loses any unfinished epoch.
To resume, restore `asr_latest.pt` and `asr_best.pt` in `RESULTS`, then set
`RESUME=True` in the training cell. `epochs` is the total budget (default 20),
not additional epochs per rerun. Test export uses the best validation checkpoint.


In [ ]:
if device.type != 'cuda':
    raise RuntimeError('Select a GPU runtime before training.')
batch_size, epochs = 2, (1 if SMOKE_TEST else 20)
train_dataset = SpeechTextDataset(DATA_DIR, 'train')
val_dataset = SpeechTextDataset(DATA_DIR, 'dev')
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, collate_fn=SpeechTextDataset.collate)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, collate_fn=SpeechTextDataset.collate)
model = ASR().to(device)
from hw4_runtime import make_asr_optimizer
optimizer = make_asr_optimizer(model, upstream_lr=1e-5, head_lr=1e-3)
best_wer, start_epoch = float('inf'), 0
RESUME = False
if RESUME and (RESULTS / 'asr_latest.pt').exists():
    state = torch.load(RESULTS / 'asr_latest.pt', map_location='cpu', weights_only=True)
    model.load_state_dict(state['model_state_dict']); optimizer.load_state_dict(state['optimizer_state_dict'])
    start_epoch, best_wer = state['epoch'] + 1, state['best_wer']
for e in range(start_epoch, epochs):
    loss = train_asr_epoch(model, train_dataloader, optimizer, device, max_steps=3 if SMOKE_TEST else None)
    val_wer = evaluate_asr(model, val_dataloader, device, max_steps=2 if SMOKE_TEST else None)
    print(f'Epoch {e+1}: loss={loss:.4f}, validation WER={val_wer:.2%}')
    if val_wer < best_wer:
        best_wer = val_wer
        save_checkpoint(RESULTS / 'asr_best.pt', model, epoch=e, best_wer=best_wer)
    save_checkpoint(RESULTS / 'asr_latest.pt', model, optimizer, epoch=e, best_wer=best_wer)

## Export test predictions
Run the exporter to write **95 `audio_id|prediction` lines** to
`RESULTS/asr_submission.txt`. Submit model outputs unchanged.

Rerun to resume an interrupted export. New model weights automatically start a
fresh file. If you change decoding code/settings, set `RESTART_PREDICTIONS=True`
once, then return it to `False` for resuming.


In [ ]:
from hw4_runtime import load_transcription
state = torch.load(RESULTS / 'asr_best.pt', map_location='cpu', weights_only=True)
model.load_state_dict(state['model_state_dict']); model.eval()
test_dataset = SpeechTextDataset(DATA_DIR, 'test')
output = RESULTS / 'asr_submission.txt'
from hw4_runtime import model_fingerprint
RESTART_PREDICTIONS = False  # True after changing decoding code/settings.
signature = model_fingerprint(model)
signature_file = RESULTS / 'asr_prediction_model.txt'
if output.exists() and (RESTART_PREDICTIONS or not signature_file.exists() or signature_file.read_text() != signature):
    output.replace(RESULTS / 'asr_submission.previous.txt')
    print('Starting new predictions; the previous file was archived.')
signature_file.write_text(signature)
predictions = load_transcription(output) if output.exists() else {}
order = (DATA_DIR / 'test_submission_order.txt').read_text().splitlines()
with torch.no_grad():
    for i in range(min(3, len(test_dataset)) if SMOKE_TEST else len(test_dataset)):
        sample = test_dataset[i]
        if predictions.get(sample['id']):
            continue
        pred, _ = model.predict(sample['wav'][None].to(device))
        predictions[sample['id']] = pred[0].replace('\n', ' ')
        temporary = output.with_suffix('.tmp')
        temporary.write_text(''.join(f'{key}|{predictions.get(key, "")}\n' for key in order))
        temporary.replace(output)
print('Saved', output)

## Report B1: CTC decoding and evaluation (3 points)
Answer in **B1** of `report.pdf` (about 3–5 sentences total, plus the sequence).
No extra training is required:

1. Decode `[A, A, blank, A, B, B, blank]` and explain the order of collapsing
   repeats and removing blanks. (1 point)
2. Why must CTC loss and decoding use valid encoded lengths to exclude padding? (1 point)
3. Why select checkpoints by validation WER rather than test WER? (1 point)


## Prepare your submission
The final cell packages your current notebook, available predictions/plots, and
**one shared `report.pdf`**. Checkpoints, datasets, and audio are excluded.
Combine separate reports into one PDF before uploading.

1. Leave `include_previous=True`. At the earlier-ZIP prompt, upload `hw4-a.zip`
   or your latest combined ZIP. For B only, cancel this prompt.
2. Upload your latest combined `report.pdf`, or cancel to keep the imported/saved
   PDF, if any.
3. Save as **`hw4-ab.zip`** for A+B, or **`hw4-b.zip`** for B alone. Submit this ZIP,
   or import it at C's earlier-ZIP prompt to add C.

**Partial work is accepted:** completed questions are graded independently.
Check the printed file list and your PDF before uploading. Every Gradescope
resubmission must contain **all** work you want graded.

**Another order or runtime reset:** set `include_previous=True` and import your
latest ZIP in whichever notebook you package last. The final cell needs no
training/setup or GPU. To include results saved on Drive, mount Drive and set
`RESULTS` to that folder's path before packaging.

Saved predictions: `RESULTS/asr_submission.txt` (95 test utterances).


In [ ]:
include_report = True  # Upload one combined PDF, or cancel to keep an imported/saved PDF.
include_previous = True  # Merge earlier submission ZIPs; cancel if none.

# Packaging needs no training dependencies, GPU, or data download.
# Course release settings are supplied by staff; no student login is needed.
COURSE_REPO_ID = 'Louis0324/eecs183-hw4'
COURSE_REVISION = 'hw4-2026-v4'
import os
os.environ.setdefault("HW4_HF_REPO", COURSE_REPO_ID)
os.environ.setdefault("HW4_HF_REVISION", COURSE_REVISION)

def ensure_hw4_support(repo_id=None, revision='hw4-2026-v4', packager_only=False):
    """Fetch the public support bundle, or use a local/uploaded offline copy."""
    import hashlib
    from io import BytesIO
    import json
    from pathlib import Path
    import re
    from urllib.parse import quote
    from urllib.request import urlopen
    import zipfile

    required = 'make_submission.py' if packager_only else 'course_download.py'
    if Path(required).is_file():
        return
    local_bundle = Path('hw4-student.zip')
    if local_bundle.is_file():
        content = local_bundle.read_bytes()
    elif repo_id:
        if not re.fullmatch(r'[\w.-]+/[\w.-]+', repo_id):
            raise ValueError('Expected a Hugging Face repository ID like COURSE_ORG/eecs183-hw4.')
        base = f'https://huggingface.co/datasets/{repo_id}/resolve/{quote(revision, safe="")}/'
        print('Downloading the small HW4 support bundle from Hugging Face; no login is required.')
        try:
            with urlopen(base + 'release_manifest.json', timeout=30) as response:
                manifest = json.loads(response.read(1024 * 1024))
            spec = manifest['files']['hw4-student.zip']
            if not 0 < spec['bytes'] < 30 * 1024 * 1024:
                raise ValueError('Invalid support bundle size in the release manifest.')
            with urlopen(base + 'hw4-student.zip', timeout=30) as response:
                content = response.read(spec['bytes'] + 1)
            if len(content) != spec['bytes'] or hashlib.sha256(content).hexdigest() != spec['sha256']:
                raise ValueError('Support bundle checksum mismatch; rerun to download a fresh copy.')
        except Exception as error:
            raise RuntimeError('Could not download the HW4 support release. Rerun this cell; '
                               'if needed, download hw4-student.zip from the course Hugging Face repo '
                               'and upload it through the Files sidebar before retrying.') from error
    else:
        # Allows local staff testing and offline/manual use before publication.
        try:
            from google.colab import files
        except ImportError as error:
            raise FileNotFoundError('Put hw4-student.zip in this folder or configure the course Hugging Face repository.') from error
        print('Upload hw4-student.zip to load the supplied helpers.')
        uploaded = files.upload()
        archives = [value for name, value in uploaded.items() if name.lower().endswith('.zip')]
        if len(archives) != 1:
            raise ValueError('Upload exactly one hw4-student.zip support bundle, then rerun.')
        content = archives[0]

    with zipfile.ZipFile(BytesIO(content)) as archive:
        if archive.testzip() is not None:
            raise ValueError('Corrupt support ZIP; download a fresh copy.')
        for item in archive.infolist():
            path = Path(item.filename)
            if (path.is_absolute() or '..' in path.parts or
                    ((item.external_attr >> 16) & 0o170000) == 0o120000 or
                    not path.resolve().is_relative_to(Path.cwd().resolve())):
                raise ValueError('Unsafe support archive path.')
            # Browser/local notebook edits must never be replaced by starter copies.
            if item.is_dir() or path.suffix == '.ipynb' or (packager_only and item.filename != required):
                continue
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(archive.read(item))
    if not Path(required).is_file():
        raise ValueError(f'This ZIP is missing {required}; use the current hw4-student.zip support bundle.')

ensure_hw4_support(COURSE_REPO_ID, COURSE_REVISION, packager_only=True)
try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None
from make_submission import build_submission, from_colab
if colab_files is not None:
    from_colab('b', include_report=include_report, include_previous=include_previous,
               results_dir=globals().get("RESULTS", "results"))
else:
    build_submission(report="report.pdf" if include_report else None)
